## Setup and Configuration

In [11]:
"""
RAFM Irradiation Analysis - Setup Cell

This notebook analyzes activation products from RAFM steel irradiation experiments,
comparing ALARA simulations against experimental gamma spectroscopy measurements.
"""

import os
import sys
import subprocess
import json
import glob
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import importlib

# ============== Directory Structure ==============
PROJECT_ROOT = Path.cwd()
ALARA_ROOT = PROJECT_ROOT.parent

# Input directories
INPUTS_DIR = PROJECT_ROOT / "alara_inputs"
MCNP_INPUTS_DIR = INPUTS_DIR / "mcnp_inputs"

# Existing data directories
EXPERIMENTAL_DIR = PROJECT_ROOT / "irradiation_QG_processed"
FLUX_WIRES_DIR = EXPERIMENTAL_DIR / "flux_wires"
SPECTRA_DIR = PROJECT_ROOT / "raw_gamma_spec"

# Scripts directory
SCRIPTS_DIR = PROJECT_ROOT / "scripts"
TOOLS_DIR = ALARA_ROOT / "tools"

# Output directories
OUTPUT_DIR = PROJECT_ROOT / "alara_output"
OUTPUT_DIR.mkdir(exist_ok=True)
(OUTPUT_DIR / 'decay_curves').mkdir(exist_ok=True)
(OUTPUT_DIR / 'activity_plots').mkdir(exist_ok=True)
(OUTPUT_DIR / 'snr_optimization').mkdir(exist_ok=True)

# MCNP Workflow directory
MCNP_WORKFLOW_DIR = ALARA_ROOT / "MCNP_ALARA_Workflow"

# ============== Python Path Setup ==============
for path in [SCRIPTS_DIR, TOOLS_DIR, MCNP_WORKFLOW_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

# ============== Import ALARA Tools ==============
from alara_output_processing import FileParser, ALARADFrame, DataLibrary, SECONDS_CONV, convert_times
from alara_output_processing import alara_output_plotting as aop_plotting
print(f"✓ ALARA output processing tools loaded")

# ============== Import Our Support Modules ==============
from nuclear_data import get_half_life, get_gamma_info, using_ensdf, get_data_source
from element_data import ELEMENT_Z, Z_TO_ELEMENT, element_to_z, z_to_element
from nuclear_data import (
    canonical_iso, format_iso_pretty, get_half_life_days, has_gamma_emission,
    get_half_life_seconds, half_life_to_lambda, parse_activity_unit, LN2, AVOGADRO
)
from nuclear_data import get_half_life_info, format_half_life

from plotting import (
    MATERIAL_COLORS, COOLING_COLORS, SOURCE_COLORS, LABELS,
    apply_standard_style, plot_all_materials_threshold, create_all_flux_plots,
)
apply_standard_style()
print(f"✓ Consolidated plotting module loaded")

import alara_comparison
print(f"✓ ALARA comparison modules loaded")

# ============== Import Experimental Data Loaders ==============
import data_loaders
importlib.reload(data_loaders)
from data_loaders import (
    load_experimental_data, parse_experimental_file,
    experimental_to_dataframe, get_top_isotopes,
    SAMPLE_TO_MATERIAL, MATERIAL_TO_TALLY, COOLING_TIME_MAP
)
from data_loaders import load_flux_wires, get_metadata_df, WIRE_METADATA
print(f"✓ Data loaders loaded")

# ============== Configuration ==============
alara_output_dir = OUTPUT_DIR

# Material names for each tally
material_names = {
    'tally_85214': 'CNA',
    'tally_85224': 'EUROFER97_C',
    'tally_85234': 'EUROFER97_B',
    'tally_85244': 'EUROFER97_A'
}

print(f"\n{'='*50}")
print(f"RAFM Irradiation Analysis - Configuration")
print(f"{'='*50}")
print(f"Project root:      {PROJECT_ROOT}")
print(f"Experimental data: {EXPERIMENTAL_DIR}")
print(f"Output dir:        {OUTPUT_DIR}")

## Load Data

Load experimental data and ALARA results for analysis.

In [12]:
# ==============================================================================
# LOAD EXPERIMENTAL DATA
# ==============================================================================
print("=" * 80)
print("LOADING EXPERIMENTAL DATA")
print("=" * 80)

experimental_data = {}

# Load RAFM3 data
exp_data_dir_3 = EXPERIMENTAL_DIR / 'RAFM3'
exp_data_3 = {}
if exp_data_dir_3.exists():
    exp_data_3 = load_experimental_data(str(exp_data_dir_3), verbose=True)
    for mat, times in exp_data_3.items():
        if mat not in experimental_data:
            experimental_data[mat] = {}
        for time_key, isotopes in times.items():
            experimental_data[mat][time_key] = isotopes
    print(f"✓ RAFM3: Loaded data for {len(exp_data_3)} materials")

# Load RAFM4 data
exp_data_dir_4 = EXPERIMENTAL_DIR / 'RAFM4'
exp_data_4 = {}
if exp_data_dir_4.exists():
    exp_data_4 = load_experimental_data(str(exp_data_dir_4), verbose=True)
    for mat, times in exp_data_4.items():
        if mat not in experimental_data:
            experimental_data[mat] = {}
        for time_key, isotopes in times.items():
            experimental_data[mat][time_key] = isotopes
    print(f"✓ RAFM4: Loaded data for {len(exp_data_4)} materials")

# Create DataFrame
if experimental_data:
    exp_df = experimental_to_dataframe(experimental_data)
    print(f"\n✓ Combined experimental data: {len(exp_df)} measurements")

# ==============================================================================
# LOAD FLUX WIRE DATA
# ==============================================================================
flux_wires_dir = FLUX_WIRES_DIR
if flux_wires_dir.exists():
    flux_wires_df = load_flux_wires(
        flux_wires_dir=str(flux_wires_dir),
        sample_filter=r'-1($|_)',
        default_irradiation_s=7200,
    )
    if not flux_wires_df.empty:
        print(f"\n✓ Loaded {len(flux_wires_df)} flux wire measurements")
else:
    flux_wires_df = pd.DataFrame()

# ==============================================================================
# LOAD ALARA ACTIVITY DATA
# ==============================================================================
print("\n" + "=" * 80)
print("LOADING ALARA RESULTS")
print("=" * 80)

activity_data = {}
for csv_file in sorted(glob.glob(os.path.join(OUTPUT_DIR, 'tally_*_Bq_per_cm3.csv'))):
    tally_name = os.path.basename(csv_file).replace('_Bq_per_cm3.csv', '')
    if tally_name == 'tally_85114':
        continue
    material = material_names.get(tally_name, 'unknown')
    df = pd.read_csv(csv_file)
    activity_data[tally_name] = {'material': material, 'data': df}
    print(f"✓ {tally_name} ({material}): {len(df)} isotopes")

# Load ALARADFrame
out_files = list(alara_output_dir.glob("**/tally_*_multizone.out"))
runs_dict = {}
for out_file in sorted(out_files):
    tally_name = out_file.parent.name
    material = material_names.get(tally_name, tally_name)
    runs_dict[material] = str(out_file)

if runs_dict:
    datalib = DataLibrary()
    adf = datalib.make_entries(runs_dict, time_unit='s')
    print(f"\n✓ Loaded ALARADFrame with {len(adf)} rows")
else:
    adf = None
    print("⚠ No ALARA output files found")

## Activity Decay Curves with Half-Life Annotations

In [13]:
# ==============================================================================
# ACTIVITY DECAY CURVES WITH HALF-LIFE ANNOTATIONS
# ==============================================================================
import nuclear_data

print("=" * 80)
print("PLOTTING ACTIVITY DECAY CURVES WITH ISOTOPE ANNOTATIONS")
print("=" * 80)

decay_curve_dir = Path(OUTPUT_DIR) / 'decay_curves'
decay_curve_dir.mkdir(parents=True, exist_ok=True)

if adf is not None and not adf.empty:
    materials = [m for m in adf['run_lbl'].unique() if m != 'tally_85114']
    
    for material in materials:
        print(f"\nProcessing: {material}")
        
        filtered = adf.filter_rows({
            'run_lbl': material,
            'variable': adf.VARIABLE_ENUM['Specific Activity']
        })
        
        piv = filtered.pivot_table(index='nuclide', columns='time', values='value', aggfunc='first')
        
        all_times = sorted([t for t in piv.columns if isinstance(t, (int, float))])
        decay_times = [t for t in all_times if t > 0]
        
        if not decay_times or 0 not in piv.columns:
            print(f"  ⚠ Insufficient time data for {material}")
            continue
        
        top10 = piv.nlargest(10, 0).index.tolist()
        
        fig = plt.figure(figsize=(14, 8))
        gs = fig.add_gridspec(1, 5, width_ratios=[4, 1, 0.1, 0.1, 0.1])
        ax = fig.add_subplot(gs[0, 0])
        
        colors = plt.cm.tab10(np.linspace(0, 1, len(top10)))
        
        isotope_info = []
        for i, iso in enumerate(top10):
            if iso not in piv.index:
                continue
            
            activities = []
            times = []
            for t in [0] + decay_times:
                if t in piv.columns:
                    val = piv.loc[iso, t]
                    if pd.notna(val) and val > 0:
                        activities.append(val)
                        times.append(t / 86400 if t > 0 else 0.001)
            
            if len(activities) > 1:
                ax.semilogy(times, activities, 'o-', color=colors[i], 
                           label=format_iso_pretty(iso), linewidth=1.5, markersize=5)
                
                hl_days, modes = get_half_life_info(iso)
                shutdown_act = activities[0] if activities else 0
                isotope_info.append({
                    'isotope': format_iso_pretty(iso),
                    'half_life': format_half_life(hl_days),
                    'shutdown_Bq_cm3': f"{shutdown_act:.2e}",
                    'color': colors[i]
                })
        
        ax.set_xlabel('Cooling Time (days)', fontsize=11)
        ax.set_ylabel('Specific Activity (Bq/cm³)', fontsize=11)
        ax.set_title(f'{material}: Decay of Top 10 Activation Products', fontsize=12, fontweight='bold')
        ax.set_xscale('log')
        ax.grid(True, alpha=0.3, which='both')
        ax.set_xlim(left=0.001)
        
        legend_labels = [f"{info['isotope']} (T½={info['half_life']})" for info in isotope_info]
        ax.legend(legend_labels, loc='upper right', fontsize=9, 
                 title='Isotope (Half-life)', title_fontsize=10)
        
        plt.tight_layout()
        
        save_path = decay_curve_dir / f'{material}_decay_curves.png'
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"  ✓ Saved: {save_path}")
        plt.show()
    
    print(f"\n{'=' * 60}")
    print(f"All decay curve plots saved to: {decay_curve_dir}")
else:
    print("⚠ ALARADFrame (adf) not available")

## Bateman Equation Decay Curves

Model activity using the Bateman equation with multi-irradiation correction.

In [14]:
# ==============================================================================
# BATEMAN EQUATION DECAY CURVES
# ==============================================================================
import decay_physics
importlib.reload(decay_physics)

from decay_physics import (
    multi_irradiation_activity, bateman_activity, get_half_life_seconds, LN2
)

print("=" * 80)
print("BATEMAN EQUATION DECAY CURVES")
print("=" * 80)

# Irradiation timeline
T_IRR1_START = 0
T_IRR1_END = 3
T_COOL = 3600 - 3
T_IRR2_START = T_IRR1_END + T_COOL
T_IRR2_END = T_IRR2_START + 7200

SCHEDULE_FULL = [
    (T_IRR1_START, T_IRR1_END, 1.0),
    (T_IRR2_START, T_IRR2_END, 1.0),
]

print(f"\nIRRADIATION TIMELINE:")
print(f"  1st irrad: t = {T_IRR1_START}-{T_IRR1_END}s (3 seconds)")
print(f"  Cooling:   t = {T_IRR1_END}-{T_IRR2_START}s (60 minutes)")
print(f"  2nd irrad: t = {T_IRR2_START}-{T_IRR2_END}s (2 hours)")

# Get experimental isotopes
if experimental_data:
    all_exp_isos = set()
    for mat_data in experimental_data.values():
        for time_data in mat_data.values():
            all_exp_isos.update(time_data.keys())
    key_isotopes = sorted(list(all_exp_isos))
else:
    key_isotopes = ['v-52', 'mn-56', 'w-187', 'cr-51', 'ta-182', 'mn-54']

valid_isotopes = [iso for iso in key_isotopes if get_half_life_seconds(iso) is not None]
print(f"\nExperimental isotopes: {len(valid_isotopes)} - {valid_isotopes}")

In [15]:
# ==============================================================================
# COLLECT EXPERIMENTAL DATA POINTS
# ==============================================================================
exp_points = {}

if experimental_data:
    material = list(experimental_data.keys())[0]
    print(f"Using experimental data from: {material}")
    
    MEASUREMENTS_AFTER_3S = {
        '300s': 300, '5min': 300,
        '2h': 7200, '2hr': 7200,
        '24h': 86400, '24hr': 86400,
        '4d': 4*86400, '4days': 4*86400,
    }
    
    MEASUREMENTS_AFTER_2HR = {
        '15d': 15*86400, '15days': 15*86400,
    }
    
    for time_key, nuclides in experimental_data[material].items():
        time_key_lower = time_key.lower()
        cooling_s = None
        reference_time = None
        
        for tk, tv in MEASUREMENTS_AFTER_3S.items():
            if tk in time_key_lower:
                cooling_s = tv
                reference_time = T_IRR1_END
                break
        
        if cooling_s is None:
            for tk, tv in MEASUREMENTS_AFTER_2HR.items():
                if tk in time_key_lower:
                    cooling_s = tv
                    reference_time = T_IRR2_END
                    break
        
        if cooling_s is None:
            continue
        
        abs_time_s = reference_time + cooling_s
        which_irrad = "3s" if reference_time == T_IRR1_END else "2hr"
        
        for iso, data in nuclides.items():
            iso_canon = canonical_iso(iso)
            activity = data.get('activity')
            unit = data.get('unit', 'uCi')
            
            if activity is not None:
                try:
                    activity_bq = float(activity) * parse_activity_unit(unit)
                    if iso_canon not in exp_points:
                        exp_points[iso_canon] = []
                    exp_points[iso_canon].append((abs_time_s, activity_bq, which_irrad))
                except:
                    pass
    
    print(f"\nCollected experimental points for {len(exp_points)} isotopes")

In [16]:
# ==============================================================================
# PLOT BATEMAN DECAY CURVES - SHORT LIVED (First 12 hours)
# ==============================================================================
# Categorize isotopes by half-life
SHORT_LIVED = []
MEDIUM_LIVED = []
LONG_LIVED = []

for iso in exp_points.keys():
    hl_s = get_half_life_seconds(iso)
    if hl_s is None:
        continue
    hl_days = hl_s / 86400
    if hl_days < 1/24:
        SHORT_LIVED.append(iso)
    elif hl_days < 10:
        MEDIUM_LIVED.append(iso)
    else:
        LONG_LIVED.append(iso)

print(f"Short-lived (T½ < 1hr): {SHORT_LIVED}")
print(f"Medium-lived (1hr-10d): {MEDIUM_LIVED}")
print(f"Long-lived (T½ > 10d): {LONG_LIVED}")

def get_scaled_activity(iso, t_array, exp_points, schedule):
    hl_s = get_half_life_seconds(iso)
    if hl_s is None:
        return None, None, None
    
    lam = LN2 / hl_s
    A_raw = multi_irradiation_activity(t_array, hl_s, schedule, production_rate=1.0)
    A_sat_raw = 1.0 / lam
    
    if iso in exp_points and exp_points[iso]:
        t_ref = exp_points[iso][0][0]
        A_ref = exp_points[iso][0][1]
        A_raw_ref = multi_irradiation_activity(t_ref, hl_s, schedule, production_rate=1.0)
        scale = A_ref / A_raw_ref if A_raw_ref > 0 else 1.0
    else:
        scale = 1e8
    
    return A_raw * scale, A_sat_raw * scale, scale

# Plot short-lived
fig, ax = plt.subplots(figsize=(12, 7))

t_zoom_days = 0.5
t_array_zoom = np.linspace(0, t_zoom_days * 86400, 2000)
t_days_zoom = t_array_zoom / 86400

short_isos = SHORT_LIVED if SHORT_LIVED else ['v-52', 'mn-56']
colors_short = plt.cm.Set1(np.linspace(0, 1, max(len(short_isos), 3)))

for i, iso in enumerate(short_isos[:8]):
    A_scaled, A_sat, scale = get_scaled_activity(iso, t_array_zoom, exp_points, SCHEDULE_FULL)
    if A_scaled is None:
        continue
    
    hl_s = get_half_life_seconds(iso)
    hl_min = hl_s / 60
    hl_label = f"{hl_min:.1f}m"
    
    ax.plot(t_days_zoom * 24 * 60, A_scaled, linewidth=2.5, color=colors_short[i % len(colors_short)],
            label=f"{format_iso_pretty(iso)} (T½={hl_label})")
    
    if iso in exp_points:
        t_exp = [(p[0] / 86400) * 24 * 60 for p in exp_points[iso] if p[0] <= t_zoom_days * 86400]
        A_exp = [p[1] for p in exp_points[iso] if p[0] <= t_zoom_days * 86400]
        if t_exp:
            ax.scatter(t_exp, A_exp, s=200, marker='o', color=colors_short[i % len(colors_short)],
                       edgecolor='black', linewidth=2, zorder=10)

ax.axvspan(T_IRR1_START/60, T_IRR1_END/60, alpha=0.3, color='green', label='1st Irrad (3s)')
ax.axvspan(T_IRR2_START/60, T_IRR2_END/60, alpha=0.3, color='red', label='2nd Irrad (2hr)')

ax.set_xlabel('Time since first irradiation (minutes)', fontsize=13, fontweight='bold')
ax.set_ylabel('Activity (Bq)', fontsize=13, fontweight='bold')
ax.set_title('Short-Lived Isotopes - First 12 Hours', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, t_zoom_days * 24 * 60)

plt.tight_layout()
plt.savefig(decay_curve_dir / 'bateman_first_half_day.png', dpi=150, bbox_inches='tight')
plt.show()

## Compare Materials

Compare total activity between different materials/tallies.

In [17]:
# ==============================================================================
# MATERIAL COMPARISON
# ==============================================================================
from flux_mesh_analysis import normalize_cooling_time

VALID_EXP_GROUPS = ['300s', '2h', '24h', '4d', '15d']

total_activity = []

for tally_name, data in activity_data.items():
    df = data['data']
    material = data['material']
    mean_cols = [c for c in df.columns if c.startswith('mean_') and '0s' not in c]
    
    for col in mean_cols:
        time_label = col.replace('mean_', '')
        normalized_time = normalize_cooling_time(time_label, exclude_shutdown=True)
        
        if normalized_time is None or normalized_time not in VALID_EXP_GROUPS:
            continue
            
        total = df[col].sum()
        total_activity.append({
            'Material': f"{material} ({tally_name})",
            'Cooling Time': normalized_time,
            'Total Activity (Bq/cm³)': total
        })

total_df = pd.DataFrame(total_activity)

if not total_df.empty:
    print("Total Activity by Material and Cooling Time:")
    print("="*80)
    
    pivot = total_df.pivot_table(
        index='Material', 
        columns='Cooling Time', 
        values='Total Activity (Bq/cm³)',
        aggfunc='first'
    )
    ordered_cols = [c for c in VALID_EXP_GROUPS if c in pivot.columns]
    pivot = pivot[ordered_cols]
    print(pivot.to_string())
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(14, 7))
    pivot.plot(kind='bar', logy=True, ax=ax)
    ax.set_xlabel('Material')
    ax.set_ylabel('Total Activity (Bq/cm³)')
    ax.set_title('Total Activity by Experimental Measurement Group')
    ax.legend(title='Cooling Time', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    
    # Save figure
    save_path = alara_output_dir / 'material_comparison.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Figure saved to: {save_path}")
    plt.show()

## Compare ALARA with Experimental Data

Compare the simulated activities with experimental measurements.

In [18]:
# ==============================================================================
# ALARA VS EXPERIMENTAL COMPARISON
# ==============================================================================
print("=" * 80)
print("ALARA VS EXPERIMENTAL COMPARISON")
print("=" * 80)

comparison_df = alara_comparison.compare_alara_to_experiment(
    experimental_data,
    alara_output_dir=alara_output_dir,
    material_to_tally=MATERIAL_TO_TALLY,
    verbose=True
)

if not comparison_df.empty:
    alara_comparison.save_comparison_csv(
        comparison_df,
        alara_output_dir / 'experimental_comparison.csv'
    )
    print(f"\nComparison DataFrame has {len(comparison_df)} rows")
else:
    print("No comparison data generated")

In [19]:
# ==============================================================================
# COMPARISON SCATTER PLOTS
# ==============================================================================
if not comparison_df.empty:
    matched = alara_comparison.get_matched_isotopes(comparison_df)
    print(f"Matched isotopes: {len(matched)}")
    
    if not matched.empty:
        alara_comparison.plot_comparison_scatter(
            matched,
            save_path=alara_output_dir / 'comparison_scatter.png'
        )
        
        material_figures = alara_comparison.plot_comparison_by_material(
            matched,
            save_dir=alara_output_dir
        )
        print(f"Generated {len(material_figures)} material comparison plots")

In [ ]:
# ==============================================================================
# ALARA THRESHOLD PLOTS
# ==============================================================================
threshold_uCi = 1e-4

# Use correct material names matching MCNP tallies
available_materials = list(MATERIAL_TO_TALLY.keys())
print(f"Available materials: {available_materials}")

# Ensure output directory exists
threshold_plot_dir = alara_output_dir / 'threshold_plots'
threshold_plot_dir.mkdir(parents=True, exist_ok=True)

figures = plot_all_materials_threshold(
    alara_output_dir=alara_output_dir,
    experimental_data=experimental_data,
    material_to_tally=MATERIAL_TO_TALLY,
    threshold_uCi=threshold_uCi,
    cooling_times=['300s', '2h', '24h', '4d', '15d'],
    save_dir=threshold_plot_dir,
    show_plots=True,  # Show plots in notebook
    show_materials=available_materials  # Show all materials
)

print(f"Generated threshold plots:")
for material, ct_figs in figures.items():
    print(f"  {material}: {len(ct_figs)} cooling time plots")
    
print(f"Threshold plots saved to: {threshold_plot_dir}")


## Flux Wire Analysis

In [ ]:
# ==============================================================================
# FLUX WIRE ANALYSIS
# ==============================================================================
from data_loaders import compute_flux_from_measurements

if 'flux_wires_df' in globals() and not flux_wires_df.empty:
    meta_df = get_metadata_df()
    
    flux_plot_df = compute_flux_from_measurements(
        flux_wires_df, 
        meta_df,
        default_irradiation_seconds=7203,
        default_decay_seconds=0
    )
    
    if not flux_plot_df.empty:
        spectrum_csv = str(SPECTRA_DIR / 'spectrum_vit_j.csv')
        figures = create_all_flux_plots(
            flux_wires_df,
            spectrum_csv=spectrum_csv,
            output_dir=OUTPUT_DIR,
            meta_df=meta_df,
            show_plots=True
        )
        print(f"\nGenerated {len(figures)} flux wire plots")
else:
    print("No flux wire data available")

## SNR Optimization for Missing Isotopes

In [ ]:
# ==============================================================================
# SNR OPTIMIZATION: HEATMAPS FOR MISSING ISOTOPES
# ==============================================================================
from matplotlib.colors import LogNorm
from decay_physics import calculate_snr_grid

print("=" * 80)
print("SNR HEATMAPS: DETECTING ALARA-PREDICTED ISOTOPES")
print("=" * 80)

snr_output_dir = Path(OUTPUT_DIR) / 'snr_optimization'
snr_output_dir.mkdir(parents=True, exist_ok=True)

def activity_to_asat(activity_bq, half_life_s, t_irradiation_s, t_cooling_s):
    """Convert measured activity to saturation activity."""
    if half_life_s <= 0 or activity_bq <= 0:
        return 0.0
    
    lam = LN2 / half_life_s
    buildup = max(1 - np.exp(-lam * t_irradiation_s), 1e-10)
    decay = max(np.exp(-lam * t_cooling_s), 1e-10)
    
    return activity_bq / (buildup * decay)

TIME_KEY_TO_COOLING_SECONDS = {
    '300s': 300, '5min': 300,
    '2h': 7200, '2hr': 7200, 
    '24h': 86400,
    '4d': 345600, '4days': 345600,
    '15d': 1296000, '15days': 1296000
}

print("\nSNR heatmaps will be generated for each material/cooldown combination.")
print(f"Output directory: {snr_output_dir}")

## Schedule Optimizer

Tools for planning irradiation + counting schedules for HPGe gamma spectroscopy.

In [ ]:
# ==============================================================================
# IMPORT SCHEDULE OPTIMIZER MODULE
# ==============================================================================
import schedule_optimizer as sopt
importlib.reload(sopt)

from schedule_optimizer import (
    hours_to_seconds, days_to_seconds, seconds_to_hours, parse_time_string,
    half_life_to_lambda, activity_at_time, integrate_activity, peak_counts,
    fwhm_model, roi_width_keV, make_efficiency_function,
    BackgroundModel, simple_continuum_model, background_counts,
    z_score, currie_Lc_Ld, mda_from_Ld, relative_uncertainty,
    GammaLine, Schedule, CountMetrics, ActivityInterpolator,
    simulate_count_for_line, simulate_schedule, compute_schedule_score,
    grid_search_schedules, format_schedule_results, print_detailed_metrics,
    create_synthetic_alara_data, create_gamma_line_table
)

print("✓ Schedule optimizer module loaded successfully")

In [ ]:
# ==============================================================================
# GAMMA LINE DATABASE FROM ENSDF
# ==============================================================================
print("=" * 70)
print("GAMMA LINE DATABASE (from ENSDF)")
print("=" * 70)

gamma_line_df = create_gamma_line_table(use_ensdf=True)

target_df = gamma_line_df[gamma_line_df['is_target']].copy()

target_lines = [
    GammaLine(
        nuclide=row['nuclide'],
        energy_keV=row['energy_keV'],
        I_gamma=row['I_gamma'],
        half_life_s=row['half_life_s'],
        is_target=row['is_target'],
        weight=row['weight'],
        notes=row['half_life_str']
    )
    for _, row in target_df.iterrows()
]

print(f"\n{'Nuclide':<10} {'E (keV)':<10} {'I_gamma':<10} {'T½':<15}")
print("-" * 50)
for line in target_lines:
    print(f"{line.nuclide:<10} {line.energy_keV:<10.1f} {line.I_gamma:<10.4f} {line.notes:<15}")

print(f"\nTotal target lines: {len(target_lines)}")

## Summary

In [ ]:
# ==============================================================================
# SUMMARY OF ANALYSIS
# ==============================================================================
print("=" * 70)
print("ALARA vs EXPERIMENTAL COMPARISON SUMMARY")
print("=" * 70)

if 'comparison_df' in globals() and not comparison_df.empty:
    matched = alara_comparison.get_matched_isotopes(comparison_df)
    unmatched = alara_comparison.get_unmatched_isotopes(comparison_df)
    
    alara_comparison.summarize_comparison(comparison_df)

print(f"\nFILES GENERATED:")
print(f"   - {alara_output_dir / 'experimental_comparison.csv'}")
print(f"   - {alara_output_dir / 'comparison_scatter.png'}")
print(f"   - Decay curve plots in {decay_curve_dir}")
print(f"   - SNR heatmaps in {snr_output_dir}")

print("\n" + "=" * 70)
print("Analysis complete!")
print("=" * 70)